# YOLO11m training on prepared cropped dataset

This notebook clones the workspace, mounts Google Drive, extracts one prepared YOLO+COCO dataset zip, validates the local layout, and starts YOLO11m training.

Expected prepared zip content can be either directly rooted at the dataset folder or nested under another folder:

```text
datasetv1_sliced_1080_2crop/
  data.yaml
  train/images/
  train/labels/
  val/images/
  val/labels/
  test/images/
  test/labels/
  annotations/train.json
  annotations/val.json
```

Supported variants:

- `datasetv1_sliced_1080_2crop`: roughly 18,000 train images, 6,500 val images
- `datasetv1_grid_740x600_6crop`: roughly 54,000 train images, 19,500 val images


In [ ]:
from pathlib import Path

# Required: set this to your workspace repo URL before running all cells.
REPO_URL = 'https://github.com/YOUR_USERNAME/obj-det-ws.git'
REPO_BRANCH = ''  # Optional. Leave empty for the default branch.

# Select which prepared Drive zip to train on.
DATASET_VARIANT = 'datasetv1_sliced_1080_2crop'  # or 'datasetv1_grid_740x600_6crop'
DATASET_DRIVE_DIR = '/content/drive/MyDrive/HYZ_2026'
PREPARED_DATASET_ZIP_DRIVE_PATH = ''  # Optional override. Empty means <DATASET_DRIVE_DIR>/<DATASET_VARIANT>_yolo_coco.zip

# Training outputs are written to Drive.
OUTPUT_DRIVE_DIR = '/content/drive/MyDrive/yolov11_runs'
RUN_NAME = f'yolo11m_{DATASET_VARIANT}'

# Weights & Biases. Leave WANDB_API_KEY empty to use Colab Secret named WANDB_API_KEY or interactive login.
USE_WANDB = True
WANDB_PROJECT = 'obj-det-ws-yolov11'
WANDB_RUN_NAME = RUN_NAME
WANDB_ENTITY = ''
WANDB_API_KEY = ''

WORKSPACE_DIR = Path('/content/obj-det-ws')
LOCAL_DATASET_EXTRACT_DIR = Path('/content/prepared_dataset_extract')

DATASET_CONFIGS = {
    'datasetv1_sliced_1080_2crop': 'configs/yolov11/yolo11m_datasetv1_sliced_1080_2crop.yaml',
    'datasetv1_grid_740x600_6crop': 'configs/yolov11/yolo11m_datasetv1_grid_740x600_6crop.yaml',
}

APPROX_DATASET_IMAGE_COUNTS = {
    'datasetv1_sliced_1080_2crop': {'train': 18000, 'val': 6500},
    'datasetv1_grid_740x600_6crop': {'train': 54000, 'val': 19500},
}

YOLO_TRAIN_PRESETS = {
    # 18k-ish train images: more epochs, still large A100 batch at fixed 640px.
    'datasetv1_sliced_1080_2crop': {'epochs': 120, 'batch': 96},
    # 54k-ish train images: fewer epochs gives a similar optimization budget.
    'datasetv1_grid_740x600_6crop': {'epochs': 80, 'batch': 96},
}

IMGSZ = 640
DEVICE = '0'
WORKERS = 8
MODEL = 'yolo11m.pt'

TRAIN_PRESET = YOLO_TRAIN_PRESETS[DATASET_VARIANT]
EPOCHS = TRAIN_PRESET['epochs']
BATCH = TRAIN_PRESET['batch']


In [ ]:
import os
import shutil
import subprocess
import sys

from google.colab import drive, userdata


def run(command, cwd=None):
    printable = ' '.join(str(part) for part in command)
    print(f'$ {printable}')
    subprocess.run([str(part) for part in command], cwd=cwd, check=True)


if 'YOUR_USERNAME' in REPO_URL:
    raise ValueError('Set REPO_URL in the parameter cell before running the notebook.')

drive.mount('/content/drive')

if WORKSPACE_DIR.exists():
    shutil.rmtree(WORKSPACE_DIR)

clone_command = ['git', 'clone', '--depth', '1', '--recurse-submodules', '--shallow-submodules']
if REPO_BRANCH:
    clone_command.extend(['--branch', REPO_BRANCH])
clone_command.extend([REPO_URL, str(WORKSPACE_DIR)])
run(clone_command)

run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'wandb', 'PyYAML', 'Pillow'])
if USE_WANDB:
    run(['yolo', 'settings', 'wandb=True'])
else:
    run(['yolo', 'settings', 'wandb=False'])
run([sys.executable, '-m', 'py_compile', WORKSPACE_DIR / 'scripts/yolov11/train.py'])


In [ ]:
def get_colab_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None


if USE_WANDB:
    import wandb

    api_key = WANDB_API_KEY or get_colab_secret('WANDB_API_KEY')
    if api_key:
        wandb.login(key=api_key)
    else:
        wandb.login()

    os.environ['WANDB_PROJECT'] = WANDB_PROJECT
    os.environ['WANDB_NAME'] = WANDB_RUN_NAME
    if WANDB_ENTITY:
        os.environ['WANDB_ENTITY'] = WANDB_ENTITY

    print(f'W&B enabled: project={WANDB_PROJECT}, run={WANDB_RUN_NAME}')
else:
    os.environ['WANDB_DISABLED'] = 'true'
    print('W&B disabled')


In [ ]:
import json
import zipfile


IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def count_files(path, suffixes=None):
    if suffixes is None:
        return sum(1 for item in path.rglob('*') if item.is_file())
    return sum(1 for item in path.rglob('*') if item.is_file() and item.suffix.lower() in suffixes)


def is_prepared_dataset_root(candidate):
    return all([
        (candidate / 'train' / 'images').is_dir(),
        (candidate / 'train' / 'labels').is_dir(),
        (candidate / 'val' / 'images').is_dir(),
        (candidate / 'val' / 'labels').is_dir(),
        (candidate / 'annotations' / 'train.json').is_file(),
        (candidate / 'annotations' / 'val.json').is_file(),
    ])


def find_prepared_dataset_root(search_root, dataset_variant):
    candidates = [search_root]
    candidates.extend(path for path in search_root.rglob('*') if path.is_dir())

    named_matches = [candidate for candidate in candidates if candidate.name == dataset_variant and is_prepared_dataset_root(candidate)]
    if named_matches:
        return named_matches[0]

    matches = [candidate for candidate in candidates if is_prepared_dataset_root(candidate)]
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise FileNotFoundError('Prepared dataset root not found. Expected train/images, train/labels, val/images, val/labels, annotations/train.json, annotations/val.json.')
    raise ValueError(f'Multiple prepared dataset roots found: {matches}. Set DATASET_VARIANT to the exact folder name.')


def warn_if_count_far(split_name, actual_count):
    expected_count = APPROX_DATASET_IMAGE_COUNTS[DATASET_VARIANT][split_name]
    diff_ratio = abs(actual_count - expected_count) / expected_count if expected_count else 0.0
    print(f'{split_name}: approx expected {expected_count}, actual {actual_count}, delta {diff_ratio:.1%}')
    if diff_ratio > 0.15:
        print(f'WARNING: {split_name} count differs from the rough expected count by more than 15%. Continuing because counts are approximate.')


def validate_prepared_dataset(dataset_root):
    actual_image_counts = {}
    for split_name in ['train', 'val']:
        images_dir = dataset_root / split_name / 'images'
        labels_dir = dataset_root / split_name / 'labels'
        image_count = count_files(images_dir, IMAGE_EXTENSIONS)
        label_count = count_files(labels_dir, {'.txt'})
        non_jpg_count = count_files(images_dir, IMAGE_EXTENSIONS - {'.jpg', '.jpeg'})
        actual_image_counts[split_name] = image_count
        print(f'{dataset_root.name}/{split_name}: {image_count} images, {label_count} labels, {non_jpg_count} non-jpg images')
        warn_if_count_far(split_name, image_count)
        if image_count == 0:
            raise ValueError(f'No images found for {split_name}: {images_dir}')
        if image_count != label_count:
            raise ValueError(f'Image/label count mismatch for {split_name}: {image_count} images, {label_count} labels')
        if non_jpg_count:
            raise ValueError(f'Expected JPG images only, found {non_jpg_count} non-JPG files in {images_dir}')

    for split_name in ['train', 'val']:
        annotation_path = dataset_root / 'annotations' / f'{split_name}.json'
        coco = json.loads(annotation_path.read_text(encoding='utf-8'))
        if not coco.get('images'):
            raise ValueError(f'No COCO images in {annotation_path}')
        if len(coco['images']) != actual_image_counts[split_name]:
            raise ValueError(f'COCO/filesystem image count mismatch for {split_name}: {len(coco["images"])} COCO images, {actual_image_counts[split_name]} files')
        for image in coco['images'][:100]:
            image_path = dataset_root / image['file_name']
            if not image_path.is_file():
                raise FileNotFoundError(f'COCO file_name does not exist: {image_path}')
        print(f'{annotation_path.name}: {len(coco["images"])} images, {len(coco["annotations"])} annotations')


if DATASET_VARIANT not in DATASET_CONFIGS:
    raise ValueError(f'Unsupported DATASET_VARIANT={DATASET_VARIANT}. Options: {sorted(DATASET_CONFIGS)}')

zip_path = Path(PREPARED_DATASET_ZIP_DRIVE_PATH) if PREPARED_DATASET_ZIP_DRIVE_PATH else Path(DATASET_DRIVE_DIR) / f'{DATASET_VARIANT}_yolo_coco.zip'
if not zip_path.is_file():
    raise FileNotFoundError(f'Prepared dataset zip not found: {zip_path}')

FINAL_DATASET_ROOT = WORKSPACE_DIR / 'datasets' / DATASET_VARIANT

for path in [LOCAL_DATASET_EXTRACT_DIR, FINAL_DATASET_ROOT]:
    if path.exists():
        shutil.rmtree(path)
LOCAL_DATASET_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)

print(f'Extracting: {zip_path}')
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(LOCAL_DATASET_EXTRACT_DIR)

extracted_dataset_root = find_prepared_dataset_root(LOCAL_DATASET_EXTRACT_DIR, DATASET_VARIANT)
print(f'extracted_dataset_root: {extracted_dataset_root}')
shutil.move(str(extracted_dataset_root), str(FINAL_DATASET_ROOT))
shutil.rmtree(LOCAL_DATASET_EXTRACT_DIR, ignore_errors=True)

validate_prepared_dataset(FINAL_DATASET_ROOT)
print(f'local_dataset_root: {FINAL_DATASET_ROOT}')


In [ ]:
CONFIG_PATH = DATASET_CONFIGS[DATASET_VARIANT]
Path(OUTPUT_DRIVE_DIR).mkdir(parents=True, exist_ok=True)

TRAIN_COMMAND = [
    sys.executable,
    'scripts/yolov11/train.py',
    '--config', CONFIG_PATH,
    '--model', MODEL,
    '--epochs', str(EPOCHS),
    '--imgsz', str(IMGSZ),
    '--batch', str(BATCH),
    '--device', DEVICE,
    '--workers', str(WORKERS),
    '--project', OUTPUT_DRIVE_DIR,
    '--name', RUN_NAME,
]

run([*TRAIN_COMMAND, '--dry-run'], cwd=WORKSPACE_DIR)


In [ ]:
run(TRAIN_COMMAND, cwd=WORKSPACE_DIR)
